<a href="https://colab.research.google.com/github/atanilson/Comp702/blob/main/Comp702_APP_MVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installing the necessary libraries

In [1]:
%%capture
#!pip -q install --upgrade folium
#!pip -q install geopandas
!pip -q install geojson
#!pip -q install geemap|
!pip -q install rasterio
#!pip -q install tqdm
#!pip -q install eeconvert

#Panel interactively
!pip install jupyter_bokeh

In [2]:
# Standard imports
import os
from tqdm.auto import tqdm
import requests
import json

import pandas as pd
import numpy as np
from PIL import Image

# Geospacial processing packages
import geopandas as gpd
import geojson

import shapely
import rasterio as rio
from rasterio.plot import show
import rasterio.mask
from shapely.geometry import box

# Mapping and plotting libraries
import matplotlib.pyplot as plt
import matplotlib.colors as cl
#import ee
#import eeconvert as eec
#import geemap
#import geemap.eefolium as emap # Commented out the old import
#import geemap.folium as emap # Added the new import
import folium

# Model
import torch
from torchvision import datasets, models, transforms

In [ ]:
# Connecting to colab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Geeting boundadies

# Area of interest and admin level
AOI = "GBR"
ADM = "ADM3"

# Query geoBoundaries
url = f"https://www.geoboundaries.org/api/current/gbOpen/{AOI}/{ADM}"

r = requests.get(url)
download_path = r.json()["gjDownloadURL"]

# Save the results as a GeoJSON
filename = "geoboundary.geojson"
geoboundary = requests.get(download_path).json()
with open(filename, "w") as file:
    geojson.dump(geoboundary, file)

In [5]:
shape_name = "Liverpool"

# Load the GeoJSON data into a GeoDataFrame
geoboundary_gdf = gpd.read_file(filename)

In [6]:
Liverpool = geoboundary_gdf.loc[geoboundary_gdf.shapeName == "Liverpool"]
Norfork = geoboundary_gdf.loc[geoboundary_gdf.shapeName == "Norfork"]
NELincolnshire = geoboundary_gdf.loc[geoboundary_gdf.shapeName == "North East Lincolnshire"]

In [ ]:
# Reading Tiles

In [8]:
save_dir = "./drive/My Drive/Colab Notebooks/Comp702/Files/"
tiles_liverpool = gpd.read_file(save_dir+"Liverpool20250712.shp")
tiles_Norfork = gpd.read_file(save_dir+"Norfork20250512.shp")
tiles_NELincolnshire = gpd.read_file(save_dir+"NELincolnshire20250711.shp")

tiles_dic = {"Liverpool":tiles_liverpool,
             "Norfork": tiles_Norfork,
             "NELincolnshire": tiles_NELincolnshire,
             }

In [9]:
# LULC Classes
classes = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake"
]

In [10]:
from typing import List
# Instantiate map centered on the centroid
def folium_Map(tiles, classes: List = classes):
  map = folium.Map(zoom_start=10,world_copy_jump=True)

  colors = {
  'AnnualCrop' : 'lightgreen',
  'Forest' : 'forestgreen',
  'HerbaceousVegetation' : 'yellowgreen',
  'Highway' : 'gray',
  'Industrial' : 'red',
  'Pasture' : 'mediumseagreen',
  'PermanentCrop' : 'chartreuse',
  'Residential' : 'magenta',
  'River' : 'dodgerblue',
  'SeaLake' : 'blue',
  'Error' : 'black'
}

  classes_plot = {}
  for classe in classes:
    classes_plot[classe] = colors[classe]

  # Add Google Satellite basemap
  folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite',
        overlay = True,
        control = True
  ).add_to(map)

  # Add LULC Map with legend
  legend_txt = '<span style="color: {col};">{txt}</span>'
  for label, color in classes_plot.items():

    # Specify the legend color
    name = legend_txt.format(txt=label, col=color)
    feat_group = folium.FeatureGroup(name=name)

    # Add GeoJSON to feature group
    subtiles = tiles[tiles.pred==label]
    if len(subtiles) > 0:
      folium.GeoJson(
          subtiles,
          style_function=lambda feature: {
            'fillColor': feature['properties']['color'],
            'color': 'black',
            'weight': 0,
            'fillOpacity': 0.5,
          },
          name='LULC Map'
      ).add_to(feat_group)
      map.add_child(feat_group)

  folium.LayerControl().add_to(map)
  map.fit_bounds(map.get_bounds())
  return map

In [11]:
import panel as pn
pn.extension()

In [15]:
def update_map(city, lista, width=700, height=700):
    # Filter the DataFrame for the selected country
    folium_map = folium_Map(tiles_dic[city], lista)
    # Panel doesn't directly render Folium maps, so we need to render it as HTML
    return pn.pane.HTML(folium_map._repr_html_(), width=width, height=height)

In [13]:
# Park Types
lulc_list = sorted(classes)
lulc_cbg = pn.widgets.CheckBoxGroup(name="lulcs", value=lulc_list, options=lulc_list)

In [14]:
# Cities Selector
cities_ls = sorted(["Liverpool","Norfork","NELincolnshire"])
cities_sl = pn.widgets.Select(name="Cities", options=cities_ls)

In [16]:
#Filsters
filters = pn.Column(
    cities_sl,
    lulc_cbg
)

In [17]:
main_map = pn.bind(update_map,cities_sl, lulc_cbg, 700, 700)

In [18]:
dashboard = pn.Row(
    filters,
    main_map
)

In [ ]:
dashboard

In [ ]:
## Grath

In [25]:
summary = tiles_liverpool["pred"].value_counts()
summary

,count
pred,
Industrial,131
Residential,130
SeaLake,110
Error,10
River,5
PermanentCrop,3
AnnualCrop,1


In [27]:
summary_df = tiles_liverpool["pred"].value_counts().reset_index()
summary_df.columns = ["pred", "tile_count"]
print(summary_df)

            pred  tile_count
0     Industrial         131
1    Residential         130
2        SeaLake         110
3          Error          10
4          River           5
5  PermanentCrop           3
6     AnnualCrop           1


In [30]:
percentages = tiles_liverpool["pred"].value_counts(normalize=True) * 100
print(percentages)

pred
Industrial       33.589744
Residential      33.333333
SeaLake          28.205128
Error             2.564103
River             1.282051
PermanentCrop     0.769231
AnnualCrop        0.256410
Name: proportion, dtype: float64


In [31]:
from math import pi

from bokeh.palettes import Category20c, Category20
from bokeh.plotting import figure
from bokeh.transform import cumsum

# Pie chat to show the selected administrative boundary number of parks
def update_chart(city):
    #filter
    tile = tiles_dic[city]
    data_plot = tile["pred"].value_counts(normalize=True) * 100
    data_plot = data_plot.reset_index(name='Percentage').rename(columns={'pred':'Landcover'})
    data_plot["angle"] = data_plot["Percentage"]/data_plot["Percentage"].sum()*2*pi
    # Assigning colors
    if len(data_plot) in Category20c:
        colors = Category20c[len(data_plot)]
    else:
        colors = Category20c[3][:len(data_plot)]  # Pick 3 colors and slice

    data_plot["color"] = colors

    data_plot['legend_label'] = data_plot['Landcover'] + ": " + round(data_plot['Percentage'],2).astype(str)+"%"

    p = figure(height=300, title = "Percenge occupied by", toolbar_location=None, width=550,
               tools="hover", tooltips="@Landcover: @Percentage", x_range=(-0.3,1.0))

    r = p.wedge(x=0, y=1, radius=0.25,
                start_angle=cumsum("angle",include_zero=True), end_angle=cumsum("angle"),
                line_color="white", fill_color="color", legend_field="legend_label", source=data_plot)

    p.axis.visible=False
    p.grid.grid_line_color = None
    p.legend.location = "right"#"top_right"
    #p.add_layout(p.legend[0], 'right')
    bokeh_pane = pn.pane.Bokeh(p, theme="dark minimal")

    return bokeh_pane


pie_chart = pn.bind(update_chart,
                   cities_sl
                   )

In [34]:
dashboard2 = pn.Row(
    cities_sl,
    pie_chart
)

In [ ]:
dashboard2